Step 1: Required librarie imported successfully

In [43]:
# time_duration
import time

In [44]:
# data analysis
import pandas as pd
import numpy as np

In [45]:
# data splitting 
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score

In [46]:
# feature selection
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2

# data balencing
from imblearn.over_sampling import SMOTE

In [47]:
#pipeline for automation
from imblearn.pipeline import Pipeline as IMBPipeline
from sklearn.pipeline import Pipeline

In [48]:
# handling Features 
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

In [49]:
# Encoding Techniques
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import MinMaxScaler

In [50]:
# Evaluation Metrics
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score

In [51]:
# Algorithms or Models
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import RidgeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import BernoulliNB

To ignore warning

In [52]:
import warnings
warnings.filterwarnings('ignore')


Loading online_shoppers_intention.csv dataset

In [53]:
print("Step 2: Created DataFrame successfully")


df = pd.read_csv("online_shoppers_intention.csv")
df.shape

Step 2: Created DataFrame successfully


(12330, 18)

Feature Engineering

In [54]:
print("Step 3: Feature Engineering Done successfully on Weekend, Revenue")


df['Weekend'] = df['Weekend'].replace((True, False), (1, 0))
df['Revenue'] = df['Revenue'].replace((True, False), (1, 0))

condition = df['VisitorType'] == 'Returning_Visitor'

Step 3: Feature Engineering Done successfully on Weekend, Revenue


In [55]:
df['Revenue'] = df['Revenue'].astype(int)

Added Returning_Visitor column

In [56]:
print("Step 4: Added Returning_Visitor column successfully")

df['Returning_Visitor'] = np.where(condition, 1, 0)

df = df.drop(columns=['VisitorType'])


Step 4: Added Returning_Visitor column successfully


Applying One Hot Encoding on Month column

In [57]:
print("Step 5: Applied one hot encoding successfully on Month column")

ordinal_encoder = OrdinalEncoder()
df['Month'] = ordinal_encoder.fit_transform(df[['Month']])

Step 5: Applied one hot encoding successfully on Month column


Checking correlation on Revenue column

In [58]:
print("Step 6: Checking correlation done successfully")

result = df[df.columns[1:]].corr()['Revenue']						
result1 = result.sort_values(ascending=False)


Step 6: Checking correlation done successfully


Prepairing Features as X and target as y

In [59]:
print("Step 7: Prepairing features as X and target as y done successfully")

X = df.drop(['Revenue'], axis=1)
y = df['Revenue']

Step 7: Prepairing features as X and target as y done successfully


In [60]:
y.unique()

array([0, 1])

In [61]:
y.dtype

dtype('int64')

Prepairing Train and Test Dataset

In [62]:
print("Step 8: Splitting data X_train, X_test, y_train & y_test done successfully")


X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size = 0.3, 
    random_state = 0
)


Step 8: Splitting data X_train, X_test, y_train & y_test done successfully


Model Pipeline

In [63]:
print("Step 9: model_pipeline fcuntion created done successfully")


def model_pipeline(X, model):

    n_c = X.select_dtypes(exclude=['object']).columns.tolist()
    c_c = X.select_dtypes(include=['object']).columns.tolist()

    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='constant')),
        ('scaler', MinMaxScaler())
    ])

    categorical_pipeline = Pipeline([
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ])

    preprocessor = ColumnTransformer([
        ('numeric', numeric_pipeline, n_c),
        ('categorical', categorical_pipeline, c_c)
    ], remainder='passthrough')

    final_steps = [
        ('preprocessor', preprocessor),
        ('smote', SMOTE(random_state=1)),
        ('feature_selection', SelectKBest(score_func = chi2, k = 6)),
        ('model', model)
    ]

    return IMBPipeline(steps = final_steps)  # Ensure to use IMBPipeline if using imblearn's Pipeline



Step 9: model_pipeline fcuntion created done successfully


 Model Selection

In [64]:
print("Step 10: select_model fcuntion created done successfully")

def select_model(X, y, pipeline=None):

    classifiers = {}
    

    c_d1 = {"RandomForestClassifier": RandomForestClassifier()}
    classifiers.update(c_d1)

    c_d2 = {"DecisionTreeClassifier": DecisionTreeClassifier()}
    classifiers.update(c_d2)

    c_d3 = {"KNeighborsClassifier": KNeighborsClassifier()}
    classifiers.update(c_d3)

    c_d4 = {"RidgeClassifier": RidgeClassifier()}
    classifiers.update(c_d4)

    c_d5 = {"BernoulliNB": BernoulliNB()}
    classifiers.update(c_d5)

    c_d6 = {"SVC": SVC()}
    classifiers.update(c_d6)
    
    
   
    cols = ['model', 'run_time', 'roc_auc']
    df_models = pd.DataFrame(columns = cols)

    for key in classifiers:
        
        start_time = time.time()
        
        print()
        print("Step 12: model_pipeline run successfully on", key)

        pipeline = model_pipeline(X, classifiers[key])
        
        cv = cross_val_score(pipeline, X, y, cv=10, scoring='roc_auc')

        row = {'model': key,
               'run_time': format(round((time.time() - start_time)/60,2)),
               'roc_auc': cv.mean(),
        }

        df_models = pd.concat([df_models, pd.DataFrame([row])], ignore_index=True)
        
    df_models = df_models.sort_values(by='roc_auc', ascending=False)
	
    return df_models


Step 10: select_model fcuntion created done successfully


Access Model select_model function

In [65]:
print("Step 11: Accessing select_model function done successfully")

models = select_model(X_train, y_train)


Step 11: Accessing select_model function done successfully

Step 12: model_pipeline run successfully on RandomForestClassifier

Step 12: model_pipeline run successfully on DecisionTreeClassifier

Step 12: model_pipeline run successfully on KNeighborsClassifier

Step 12: model_pipeline run successfully on RidgeClassifier

Step 12: model_pipeline run successfully on BernoulliNB

Step 12: model_pipeline run successfully on SVC


Lets see total model with score

In [66]:
print("Step 13: Accessing select_model function done successfully")
print()

print(models)


Step 13: Accessing select_model function done successfully

                    model run_time   roc_auc
5                     SVC      1.9  0.889927
0  RandomForestClassifier     0.62  0.887913
4             BernoulliNB     0.02  0.870663
3         RidgeClassifier     0.02  0.856459
2    KNeighborsClassifier     0.03  0.840885
1  DecisionTreeClassifier     0.04  0.736522


Accessing best model and training

In [67]:
print()
print("Step 14: Accessing select_model function done successfully")

selected_model = SVC()
bundled_pipeline = model_pipeline(X_train, selected_model)
bundled_pipeline.fit(X_train, y_train)



Step 14: Accessing select_model function done successfully


,steps,"[('preprocessor', ...), ('smote', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready 

Accessing best model and training

In [68]:
print("Step 15: Results predicted successfully")

y_pred = bundled_pipeline.predict(X_test)



Step 15: Results predicted successfully


ROC and AOC score

In [69]:
print("Step 16: ROC and AOC scores")

roc_auc = roc_auc_score(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)
f1_score = f1_score(y_test, y_pred)

print()
print('ROC/AUC:', roc_auc)
print('Accuracy:', accuracy)
print('F1 score:', f1_score)
print()


Step 16: ROC and AOC scores

ROC/AUC: 0.7872136596906621
Accuracy: 0.8775344687753447
F1 score: 0.6413301662707839



Classification report

In [70]:
print("Step 17: classification report generated successfully")

classif_report = classification_report(y_test, y_pred)

print(classif_report)


Step 17: classification report generated successfully
              precision    recall  f1-score   support

           0       0.93      0.92      0.93      3077
           1       0.63      0.65      0.64       622

    accuracy                           0.88      3699
   macro avg       0.78      0.79      0.78      3699
weighted avg       0.88      0.88      0.88      3699

